## 1 · Setup
Clones the repository, checks `ffmpeg` and installs the dependencies (NeMo is skipped on purpose — it conflicts with `qwen-asr`).

| Option | Values |
|---|---|
| `UPDATE_REPO` | `True` pull the latest code on re-run · `False` keep local edits |

In [ ]:
import gc
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ.setdefault("USE_TF", "0")

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
REPO_URL = "https://github.com/CaptainLevi04/Audio-Datasets-Creation-System.git"
UPDATE_REPO = True

if (WORK_ROOT / "speech_corpus").is_dir():
    REPO_DIR = WORK_ROOT
elif (Path.cwd() / "speech_corpus").is_dir():
    REPO_DIR = Path.cwd()
else:
    REPO_DIR = WORK_ROOT / "Audio-Datasets-Creation-System"
    if not REPO_DIR.exists():
        print(f"📥 Cloning repository into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif UPDATE_REPO and (REPO_DIR / ".git").exists():
        print(f"🔄 Updating repository in {REPO_DIR} ...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

os.chdir(WORK_ROOT)
if str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

if shutil.which("ffmpeg") is None:
    print("ffmpeg not found on PATH; attempting installation ...")
    if sys.platform.startswith("linux"):
        subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg", shell=True, check=False)
    if shutil.which("ffmpeg") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"])

%pip install -q "yt-dlp[default]" qwen-asr librosa soundfile jiwer faster-whisper tqdm ipywidgets

### Imports & helpers
Loads the `speech_corpus` modules, detects the compute device and defines two small shared helpers.

In [ ]:
import jiwer
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

from speech_corpus import ast_model, segmentation
from speech_corpus.ast_model import classify_audio, save_regions, validate_cached_regions
from speech_corpus.audio import DEFAULT_SR, read_wav, write_wav
from speech_corpus.cut_engine_v2 import cut_speech_and_mixed_segments
from speech_corpus.downloader import (
    download_video, extract_video_id, find_cached_video, ingest_local_file,
)
from speech_corpus.quality_control import (
    generate_qc_markdown_report, split_corpus, validate_segments_integrity, video_summary,
)
from speech_corpus.segmentation import detect_speech_activity_overlay
from speech_corpus.snr import compute_snr_for_segments, estimate_segment_snr, plot_snr_histogram
from speech_corpus.visualization import plot_segments_overview, plot_spectrogram_regions

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

device = "cuda" if torch.cuda.is_available() else "cpu"
dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"


def banner(title, char="="):
    print(char * 72)
    print(title)
    print(char * 72)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"✅ Environment ready | ON_KAGGLE={ON_KAGGLE} | WORK_ROOT={WORK_ROOT}")
print(f"   Repo: {REPO_DIR} | Device: {device.upper()} ({dev_name})")

## 2 · Configuration
The only cell you need to edit — pass one or more videos (YouTube URLs or local files) and tune the options below.

| Option | Possible values |
|---|---|
| `SOURCES` | `"https://youtube.com/watch?v=..."` · `{"url": "...", "video_id": "lecture1"}` · `{"file": "/kaggle/input/<dataset>/lecture.mp4", "video_id": "lecture1"}` |
| `USE_MOCK_DEMO` | `True` runs an offline synthetic smoke test and ignores `SOURCES` |
| `COOKIES_FILE` | `None` · path to a YouTube `cookies.txt` |
| `ATTACHED_OUTPUT_ROOT` | `None` · `Path("/kaggle/input/<dataset>/corpus_data")` to reuse cached outputs |
| `FORCE_RECLASSIFY` / `FORCE_RECUT` / `FORCE_RESNR` / `FORCE_RETRANSCRIBE` | `True` ignores the cache and recomputes that stage |
| `ACCEPT_QC_FAILED_SEGMENTS` | `True` continues when Cut Engine QC fails (segments are promoted from `segments_staging/`) · `False` stops with an error |
| `USE_WORD_VERIFICATION` | `True` enables the faster-whisper word-boundary check during cutting (`WORD_MODEL_SIZE`: `"small"`, `"medium"`, …) |
| `SNR_THRESHOLD_DB` | clean/noisy cut-off in dB — `25.0` default, the spec starts at `20.0` |
| `SNR_COMBINE_MODE` | `"mean"` · `"wada"` · `"energy"` · `"min"` |
| `ASR_LIMIT` | `None` for all `speech`/`mixed` segments · e.g. `10` for a quick test |
| `WHISPER_BATCH` / `AUDAR_BATCH` | lower to `2`–`1` / `4` on CUDA out-of-memory |
| `ASR_ACCEPT_CER` / `ASR_HUMAN_MAX_WORD_DIFF` | agreement rule: CER ≤ first → `accepted`; word diffs ≤ second → `human_choice`; otherwise `difficult` |
| `SPLIT_RATIOS` | `(train, val, test)` — splits are created only with ≥ 3 videos |

In [ ]:
SOURCES = [
    {"url": "https://www.youtube.com/watch?v=QQd1lBreIaA"},
]

USE_MOCK_DEMO = False
COOKIES_FILE = None
ATTACHED_OUTPUT_ROOT = None

DATA_ROOT = WORK_ROOT / "corpus_data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
SR = DEFAULT_SR

SILENCE_THRESHOLD_S = 0.25
FORCE_RECLASSIFY = False
FORCE_RECUT = False
ACCEPT_QC_FAILED_SEGMENTS = True
USE_WORD_VERIFICATION = False
WORD_MODEL_SIZE = "small"

SNR_THRESHOLD_DB = 25.0
SNR_COMBINE_MODE = "mean"
FORCE_RESNR = False

ASR_LABELS = ("speech", "mixed")
ASR_LIMIT = None
FORCE_RETRANSCRIBE = False

WHISPER_ID = "openai/whisper-large-v3"
WHISPER_LANG = "ar"
WHISPER_BATCH = 4
WHISPER_BEAMS = 5

AUDAR_ID = "audarai/Audar-ASR-V1-Turbo"
AUDAR_LANG = "Arabic"
AUDAR_BATCH = 8

ASR_ACCEPT_CER = 0.02
ASR_HUMAN_MAX_WORD_DIFF = 3
SPLIT_RATIOS = (0.70, 0.15, 0.15)

if ATTACHED_OUTPUT_ROOT is not None:
    src_root = Path(ATTACHED_OUTPUT_ROOT)
    if src_root.exists():
        for item in src_root.iterdir():
            dst = DATA_ROOT / item.name
            if item.is_dir() and not dst.exists():
                shutil.copytree(item, dst)
        print(f"📂 Copied cached outputs from {src_root} -> {DATA_ROOT.resolve()}")

print(f"⚙️  Configured {len(SOURCES)} source(s) | USE_MOCK_DEMO={USE_MOCK_DEMO} | DATA_ROOT={DATA_ROOT.resolve()}")

## 3 · Phase 1 — Download, Label & Segment
Defines the source resolver, audio ingestion and the QC-recovery step that promotes `segments_staging/` to `segments/`.

In [ ]:
def resolve_local_file(file_str):
    p = Path(file_str)
    if p.exists():
        return p
    if Path("/kaggle/input").exists():
        matches = list(Path("/kaggle/input").rglob(p.name))
        if matches:
            print(f"Resolved local file from Kaggle input: {matches[0]}")
            return matches[0]
    raise FileNotFoundError(f"Could not find media file: {file_str}")


def prepare_jobs():
    if USE_MOCK_DEMO:
        return [{"mode": "mock", "video_id": "mock_demo_audio"}]
    jobs = []
    for idx, entry in enumerate(SOURCES, 1):
        if isinstance(entry, str):
            entry = {"url": entry}
        if entry.get("file"):
            fp = resolve_local_file(entry["file"])
            vid = entry.get("video_id") or extract_video_id(str(fp))
            jobs.append({"mode": "file", "file": fp, "video_id": vid})
        elif entry.get("url"):
            vid = entry.get("video_id") or extract_video_id(entry["url"])
            jobs.append({"mode": "url", "url": entry["url"], "video_id": vid})
        else:
            raise ValueError(f"Invalid SOURCES entry #{idx}: {entry}")
    return jobs


def ingest_audio(job, video_dir):
    audio_path = video_dir / "audio.wav"
    info_path = video_dir / "info.json"

    if audio_path.exists() and info_path.exists():
        print(f"Found cached audio.wav & info.json in {video_dir}")
        info = json.loads(info_path.read_text(encoding="utf-8"))
    elif job["mode"] == "mock":
        from speech_corpus.labelling import build_mock_audio
        samples, _ = build_mock_audio(seed=42, sr=SR)
        write_wav(audio_path, samples, sample_rate=SR)
        info = {
            "video_id": video_dir.name, "source": "mock", "url": None,
            "title": "Synthetic Audio Demo", "channel": "Synthesizer",
            "duration_s": round(len(samples) / SR, 2),
            "license": "Creative Commons Attribution", "is_creative_commons": True,
        }
        info_path.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
        return samples, info, video_dir
    elif job["mode"] == "file":
        audio_path, info = ingest_local_file(job["file"], video_dir)
    else:
        cached = find_cached_video(job["url"], search_dirs=[DATA_ROOT])
        if cached is not None:
            audio_path, info = cached
            video_dir = audio_path.parent
        else:
            audio_path, info = download_video(job["url"], out_dir=video_dir, cookies_file=COOKIES_FILE)

    samples, _ = read_wav(audio_path, target_sr=SR)
    return samples, info, video_dir


def finalize_segments(segments_df, video_dir):
    segments_dir = video_dir / "segments"
    staging_dir = video_dir / "segments_staging"

    if not segments_dir.exists() and staging_dir.exists():
        if not ACCEPT_QC_FAILED_SEGMENTS:
            raise RuntimeError(
                f"Cut Engine v2 QC failed for {video_dir.name}; segments were left in {staging_dir}. "
                "Fix the QC errors or set ACCEPT_QC_FAILED_SEGMENTS = True."
            )
        staging_dir.rename(segments_dir)
        for col in ("audio_file", "wav_file"):
            if col in segments_df.columns:
                segments_df[col] = segments_df[col].map(
                    lambda x: x.replace("segments_staging/", "segments/") if isinstance(x, str) else x
                )
        print(f"⚠️  QC failed for {video_dir.name}: promoted segments_staging/ -> segments/ "
              f"(see {video_dir / 'qc_report.md'} for the failed gates)")
    elif not segments_dir.exists():
        raise RuntimeError(f"Cut Engine v2 produced no segments for {video_dir.name}")

    segments_df.to_csv(video_dir / "segments_student1.csv", index=False)
    return segments_df

Defines the per-video pipeline: ingest → 16 kHz WAV → VAD pauses → AST labels → Cut Engine v2 → 10-gate QC.

In [ ]:
def run_student1(job, idx, total):
    vid = job["video_id"]
    video_dir = DATA_ROOT / vid
    video_dir.mkdir(parents=True, exist_ok=True)
    banner(f"[Phase 1 | Video {idx}/{total}] video_id={vid} -> {video_dir}")

    audio, info, video_dir = ingest_audio(job, video_dir)
    dur_s = len(audio) / SR
    print(f"Audio ready: {dur_s:.1f}s ({dur_s / 60:.2f} min) @ {SR} Hz | License: {info.get('license')}")

    backup_csv = video_dir / "segments_student1.csv"
    regions_csv = video_dir / "regions.csv"
    windows_csv = video_dir / "windows_ast.csv"

    _, pauses_df = detect_speech_activity_overlay(
        audio=audio, sr=SR,
        silence_threshold_s=SILENCE_THRESHOLD_S,
        min_pause_ms=120, vad_backend="auto",
    )

    if (not FORCE_RECLASSIFY) and validate_cached_regions(
        regions_csv, expected_source_id=video_dir.name, expected_duration=dur_s, expected_version="2.2"
    ):
        print(f"Loaded cached regions (v2.2): {regions_csv}")
        regions_df = pd.read_csv(regions_csv)
    else:
        print("Running AST macro-region classification ...")
        regions_df, windows_df = classify_audio(
            y=audio, sr=SR, win_s=10.0, hop_s=5.0, batch_size=32, pauses_df=pauses_df
        )
        save_regions(regions_df, regions_csv, source_id=video_dir.name, duration=dur_s, version="2.2")
        if windows_df is not None and not windows_df.empty:
            windows_df.to_csv(windows_csv, index=False)

    spec_path = video_dir / "full_spectrogram.png"
    if not spec_path.exists() or FORCE_RECLASSIFY:
        fig = plot_spectrogram_regions(
            audio, regions_df, sr=SR,
            title=f"Full Audio Spectrogram - {str(info.get('title', vid))[:40]}",
            save_path=spec_path,
        )
        plt.close(fig)

    if (not FORCE_RECUT) and backup_csv.exists() and (video_dir / "segments").exists():
        segments_df = pd.read_csv(backup_csv, dtype={"segment_id": str})
        print(f"Loaded cached segments ({len(segments_df)} rows): {backup_csv}")
    else:
        segments_df = cut_speech_and_mixed_segments(
            audio=audio,
            regions_df=regions_df,
            out_dir=video_dir,
            sr=SR,
            pauses=pauses_df,
            use_word_verification=USE_WORD_VERIFICATION,
            word_model_size=WORD_MODEL_SIZE,
            run_ablation=True,
        )
        segments_df = finalize_segments(segments_df, video_dir)

    overview_path = video_dir / "segments_overview.png"
    if not overview_path.exists() or FORCE_RECUT:
        fig = plot_segments_overview(audio, segments_df, sr=SR, max_segments=20, save_path=overview_path)
        plt.close(fig)

    qc = validate_segments_integrity(segments_df, video_dir=video_dir, total_dur=dur_s, sr=SR)
    generate_qc_markdown_report(segments_df, report_path=video_dir / "qc_report.md", video_dir=video_dir, qc=qc)
    vsum_df = video_summary(regions_df=regions_df, segments_df=segments_df, info=info, video_dir=video_dir)

    lbl_counts = segments_df["label"].astype(str).str.lower().value_counts().to_dict()
    print(f"QC status: {qc['status']} ({qc['summary']['passed_gates']}/{qc['summary']['total_gates']} gates)")
    print(f"Segments: {len(segments_df)} total | By label: {lbl_counts}")
    display(vsum_df)

    del audio
    gc.collect()
    return video_dir

Runs Phase 1 for every source and saves `audio.wav`, `regions.csv`, `segments/*.wav`, `qc_report.md` and `segments_student1.csv`.

In [ ]:
jobs = prepare_jobs()
video_dirs = [run_student1(job, i, len(jobs)) for i, job in enumerate(jobs, 1)]

ast_model._AST_CACHE.clear()
segmentation._SILERO.clear()
free_gpu()
print(f"\n✅ Phase 1 complete. Outputs saved under: {DATA_ROOT.resolve()}")

## 4 · Phase 2 — SNR Estimation
Computes Energy and WADA SNR for **all** segments and saves `segments_snr.csv`, `snr_diagnostics.csv` and both histograms.

| Option | Possible values |
|---|---|
| `SNR_THRESHOLD_DB` | clean/noisy cut-off in dB (default `25.0`) |
| `SNR_COMBINE_MODE` | `"mean"` · `"wada"` · `"energy"` · `"min"` |
| `FORCE_RESNR` | `True` recomputes even when cached |

In [ ]:
def save_length_histogram(durations, path, title):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(durations.dropna(), bins=25, color="#2ca02c", edgecolor="black", alpha=0.85)
    ax.axvline(5.0, color="red", ls="--", label="5s min")
    ax.axvline(15.0, color="red", ls="--", label="15s max")
    ax.set_xlabel("Segment duration (s)")
    ax.set_ylabel("Number of segments")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    plt.close(fig)


snr_dirs = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and (p / "segments").is_dir()
    and ((p / "segments_student1.csv").exists() or (p / "segments.csv").exists())
)
assert snr_dirs, f"No segmented videos found in {DATA_ROOT.resolve()}. Run Phase 1 first."

for video_dir in snr_dirs:
    banner(f"[Phase 2 | SNR] {video_dir}")

    s1_csv = video_dir / "segments_student1.csv"
    src_csv = s1_csv if s1_csv.exists() else (video_dir / "segments.csv")
    seg_src_df = pd.read_csv(src_csv, dtype={"segment_id": str})

    if "wav_file" not in seg_src_df.columns and "audio_file" in seg_src_df.columns:
        seg_src_df["wav_file"] = seg_src_df["audio_file"].map(
            lambda x: f"segments/{Path(str(x)).name}" if pd.notnull(x) and not (video_dir / str(x)).exists() else x
        )

    lbl_counts = seg_src_df["label"].astype(str).str.lower().value_counts().to_dict()
    print(f"Passing ALL {len(seg_src_df)} segments to SNR: {lbl_counts}")

    snr_ckpt = video_dir / "segments_snr.csv"
    diag_csv = video_dir / "snr_diagnostics.csv"

    if (not FORCE_RESNR) and snr_ckpt.exists() and diag_csv.exists():
        snr_df = pd.read_csv(snr_ckpt, dtype={"segment_id": str})
        print(f"Loaded cached SNR results: {snr_ckpt}")
    else:
        snr_df, _ = compute_snr_for_segments(
            seg_src_df,
            base_dir=video_dir,
            threshold_db=SNR_THRESHOLD_DB,
            combine=SNR_COMBINE_MODE,
            save_csv=True,
            verbose=True,
        )
        snr_df.to_csv(snr_ckpt, index=False, encoding="utf-8-sig")

        diag_rows = []
        for _, row in seg_src_df.iterrows():
            rel_wav = row["wav_file"] if "wav_file" in seg_src_df.columns else row["audio_file"]
            wav_path = video_dir / Path(str(rel_wav))
            if not wav_path.exists():
                wav_path = video_dir / "segments" / Path(str(rel_wav)).name
            samples, sr_w = read_wav(wav_path, target_sr=SR)
            est = estimate_segment_snr(samples, sr=sr_w, threshold_db=SNR_THRESHOLD_DB, combine=SNR_COMBINE_MODE)
            diag_rows.append({
                "segment_id": row["segment_id"],
                "label": row["label"],
                "snr_energy_db": est["snr_energy_db"],
                "snr_wada_db": est["snr_wada_db"],
                "snr": est["snr"],
                "quality": est["quality"],
                "snr_disagree": est["snr_disagree"],
            })
        pd.DataFrame(diag_rows).to_csv(diag_csv, index=False, encoding="utf-8-sig")

    fig = plot_snr_histogram(snr_df, threshold_db=SNR_THRESHOLD_DB, save_path=video_dir / "snr_histogram.png")
    plt.close(fig)

    durations = pd.to_numeric(snr_df["end"], errors="coerce") - pd.to_numeric(snr_df["start"], errors="coerce")
    save_length_histogram(
        durations, video_dir / "segment_length_histogram.png",
        f"Segment-Length Distribution ({video_dir.name})",
    )

    mean_snr = pd.to_numeric(snr_df["snr"], errors="coerce").mean()
    print(f"Quality counts: {snr_df['quality'].value_counts().to_dict()} | Mean SNR: {mean_snr:.2f} dB")
    display(snr_df[["segment_id", "start", "end", "label", "snr", "quality"]].head(8))

print("\n✅ Phase 2 complete.")

## 5 · Phase 3 — Dual ASR
Keeps only `speech`/`mixed` segments (music and silence skip ASR) and loads their audio into memory.

| Option | Possible values |
|---|---|
| `ASR_LABELS` | labels sent to ASR — default `("speech", "mixed")` |
| `ASR_LIMIT` | `None` for all segments · e.g. `10` for a quick test |

In [ ]:
def resolve_segment_wav(video_dir, raw_path):
    p = Path(str(raw_path))
    candidates = [
        p if p.is_absolute() else None,
        WORK_ROOT / p,
        video_dir / p,
        video_dir / "segments" / p.name,
        video_dir / "segments_staging" / p.name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return c
    raise FileNotFoundError(f"Missing segment WAV '{raw_path}' for {video_dir}")


def load_asr_checkpoint(vdir, df_asr, ckpt_name, column):
    ckpt = vdir / ckpt_name
    if FORCE_RETRANSCRIBE or not ckpt.exists():
        return False
    c_df = pd.read_csv(ckpt, dtype={"segment_id": str})
    same_rows = len(c_df) == len(df_asr) and [str(x) for x in c_df["segment_id"]] == [str(x) for x in df_asr["segment_id"]]
    if not same_rows:
        return False
    df_asr[column] = c_df[column].fillna("").tolist()
    return True


asr_dirs = sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and (p / "segments_snr.csv").exists())
assert asr_dirs, f"No segments_snr.csv found under {DATA_ROOT.resolve()}. Run Phase 2 first."

allowed_labels = {s.lower() for s in ASR_LABELS}
asr_jobs = {}

for video_dir in asr_dirs:
    snr_df = pd.read_csv(video_dir / "segments_snr.csv", dtype={"segment_id": str})
    snr_df["_row_idx"] = np.arange(len(snr_df))
    lbl_norm = snr_df["label"].astype(str).str.strip().str.lower()

    asr_mask = lbl_norm.isin(allowed_labels)
    df_asr_all = snr_df[asr_mask].copy().reset_index(drop=True)
    df_non_asr = snr_df[~asr_mask].copy().reset_index(drop=True)

    if ASR_LIMIT is not None:
        df_asr = df_asr_all.head(ASR_LIMIT).copy().reset_index(drop=True)
        df_asr_skipped = df_asr_all.iloc[ASR_LIMIT:].copy().reset_index(drop=True)
    else:
        df_asr = df_asr_all
        df_asr_skipped = pd.DataFrame(columns=snr_df.columns)

    if not df_asr.empty:
        bad = set(df_asr["label"].astype(str).str.strip().str.lower()) - allowed_labels
        assert not bad, f"Unexpected labels passed to ASR: {bad}"

    print(
        f"[{video_dir.name}] Total={len(snr_df)} | ASR={len(df_asr)} | Non-ASR={len(df_non_asr)}"
        + (f" | Deferred(ASR_LIMIT)={len(df_asr_skipped)}" if len(df_asr_skipped) else "")
    )

    audio_list = []
    for row in df_asr.itertuples():
        x, _ = read_wav(resolve_segment_wav(video_dir, row.audio_file), target_sr=SR)
        audio_list.append(x)

    durs = np.array([len(a) / SR for a in audio_list], dtype=float)
    if len(durs):
        assert durs.max() <= 30.5, (
            f"[{video_dir.name}] Segment exceeds 30.5s (max={durs.max():.2f}s); ASR models support <= 30s."
        )
        print(f"  Loaded {len(audio_list)} clips ({durs.sum() / 60:.1f} min, max={durs.max():.1f}s)")
    rms = np.array([float(np.sqrt((a ** 2).mean())) for a in audio_list], dtype=float)

    asr_jobs[video_dir] = {
        "df_asr": df_asr,
        "df_non_asr": df_non_asr,
        "df_asr_skipped": df_asr_skipped,
        "audio": audio_list,
        "rms": rms,
    }

**Model A — Whisper large-v3:** transcribes the clips, saves `asr_whisper_ckpt.csv` per video and frees the GPU.

| Option | Possible values |
|---|---|
| `WHISPER_BATCH` | `4` default — lower to `2` or `1` on CUDA out-of-memory |
| `WHISPER_BEAMS` / `WHISPER_LANG` | beam-search width · language code (`"ar"`) |
| `FORCE_RETRANSCRIBE` | `True` ignores existing checkpoints |

In [ ]:
need_whisper = []
for vdir, job in asr_jobs.items():
    df_asr = job["df_asr"]
    if df_asr.empty:
        df_asr["model_A_transcript"] = []
    elif load_asr_checkpoint(vdir, df_asr, "asr_whisper_ckpt.csv", "model_A_transcript"):
        print(f"[{vdir.name}] Loaded cached Whisper checkpoint")
    else:
        need_whisper.append(vdir)

if need_whisper:
    from transformers import AutoProcessor, WhisperForConditionalGeneration

    w_dev = "cuda" if torch.cuda.is_available() else "cpu"
    w_dtype = torch.float16 if w_dev == "cuda" else torch.float32
    print(f"Loading {WHISPER_ID} on {w_dev} (dtype={w_dtype}) ...")
    w_proc = AutoProcessor.from_pretrained(WHISPER_ID)
    w_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID, torch_dtype=w_dtype).to(w_dev).eval()

    for vdir in need_whisper:
        job = asr_jobs[vdir]
        audio_list, rms = job["audio"], job["rms"]
        whisper_txt = []
        print(f"Transcribing {vdir.name} ({len(audio_list)} segments) with Whisper large-v3 ...")
        for i in tqdm(range(0, len(audio_list), WHISPER_BATCH)):
            batch = audio_list[i:i + WHISPER_BATCH]
            feats = w_proc(batch, sampling_rate=SR, return_tensors="pt").input_features.to(w_dev, dtype=w_dtype)
            with torch.inference_mode():
                ids = w_model.generate(
                    feats, language=WHISPER_LANG, task="transcribe",
                    num_beams=WHISPER_BEAMS, max_new_tokens=300,
                )
            texts = w_proc.batch_decode(ids, skip_special_tokens=True)
            for k, t in enumerate(texts):
                whisper_txt.append("" if rms[i + k] < 1e-3 else t.strip())

        job["df_asr"]["model_A_transcript"] = whisper_txt
        job["df_asr"][["_row_idx", "segment_id", "label", "model_A_transcript"]].to_csv(
            vdir / "asr_whisper_ckpt.csv", index=False, encoding="utf-8-sig"
        )
        print(f"Saved Whisper checkpoint: {vdir / 'asr_whisper_ckpt.csv'}")

    del w_model, w_proc
    free_gpu()
    if torch.cuda.is_available():
        print(f"Freed GPU after Whisper | Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Whisper skipped (everything cached or no speech/mixed segments).")

**Model B — Audar-ASR-V1-Turbo:** transcribes the same clips, saves `asr_audar_ckpt.csv` per video and frees the GPU.

| Option | Possible values |
|---|---|
| `AUDAR_BATCH` | `8` default — lower to `4` on CUDA out-of-memory |
| `AUDAR_LANG` | language name passed to the model (`"Arabic"`) |

In [ ]:
need_audar = []
for vdir, job in asr_jobs.items():
    df_asr = job["df_asr"]
    if df_asr.empty:
        df_asr["model_B_transcript"] = []
    elif load_asr_checkpoint(vdir, df_asr, "asr_audar_ckpt.csv", "model_B_transcript"):
        df_asr["second_model"] = "audar"
        print(f"[{vdir.name}] Loaded cached Audar checkpoint")
    else:
        need_audar.append(vdir)

if need_audar:
    from qwen_asr import Qwen3ASRModel

    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability(0)
        a_dtype = torch.bfloat16 if major >= 8 else torch.float16
        a_device = "cuda:0"
    else:
        a_dtype = torch.float32
        a_device = "cpu"

    print(f"Loading {AUDAR_ID} on {a_device} (dtype={a_dtype}) ...")
    a_model = Qwen3ASRModel.from_pretrained(
        AUDAR_ID, dtype=a_dtype, device_map=a_device,
        max_inference_batch_size=AUDAR_BATCH, max_new_tokens=256,
    )
    a_model.model.to(a_dtype)

    for vdir in need_audar:
        job = asr_jobs[vdir]
        audio_list, rms = job["audio"], job["rms"]
        audar_txt = []
        print(f"Transcribing {vdir.name} ({len(audio_list)} segments) with Audar-ASR-V1-Turbo ...")
        for i in tqdm(range(0, len(audio_list), AUDAR_BATCH)):
            batch = [(a, SR) for a in audio_list[i:i + AUDAR_BATCH]]
            out = a_model.transcribe(audio=batch, language=[AUDAR_LANG] * len(batch))
            audar_txt.extend(r.text.strip() for r in out)

        job["df_asr"]["second_model"] = "audar"
        job["df_asr"]["model_B_transcript"] = [("" if r < 1e-3 else t) for t, r in zip(audar_txt, rms)]
        job["df_asr"][["_row_idx", "segment_id", "label", "model_B_transcript"]].to_csv(
            vdir / "asr_audar_ckpt.csv", index=False, encoding="utf-8-sig"
        )
        print(f"Saved Audar checkpoint: {vdir / 'asr_audar_ckpt.csv'}")

    del a_model
    free_gpu()
    if torch.cuda.is_available():
        print(f"Freed GPU after Audar | Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Audar skipped (everything cached or no speech/mixed segments).")

Defines Arabic text normalisation and the agreement rule between the two transcripts.

| Option | Possible values |
|---|---|
| `ASR_ACCEPT_CER` | CER at or below this → `accepted` (default `0.02`) |
| `ASR_HUMAN_MAX_WORD_DIFF` | word differences at or below this → `human_choice`, otherwise `difficult` (default `3`) |

In [ ]:
DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
ALEF_MAP = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"})

APP_COLS = [
    "start", "end", "label", "snr", "quality",
    "model_A_transcript", "model_B_transcript", "final_transcript",
    "status", "review_status", "automatic_status",
    "segment_id", "audio_file",
    "changed_by", "changed_at", "change_log",
]

BLANK_META = {
    "model_A_transcript": "", "model_B_transcript": "", "final_transcript": "",
    "cer": np.nan, "n_diff_words": np.nan, "second_model": "",
    "changed_by": "", "changed_at": "", "change_log": "[]",
}


def normalize_ar(t):
    t = "" if t is None or (isinstance(t, float) and t != t) else str(t)
    t = DIACRITICS.sub("", t).replace("ـ", "").translate(ALEF_MAP)
    t = t.replace("ة", "ه").replace("ى", "ي")
    t = re.sub(r"[^\w\s]", " ", t).replace("_", " ")
    return " ".join(t.split())


def compare_transcripts(a, b):
    na, nb = normalize_ar(a), normalize_ar(b)
    if not na or not nb:
        return "difficult", 1.0, 999
    w = jiwer.process_words(na, nb)
    n_diff = int(w.substitutions + w.deletions + w.insertions)
    cer = float(jiwer.cer(na, nb))
    if cer <= ASR_ACCEPT_CER:
        return "accepted", round(cer, 4), n_diff
    status = "human_choice" if n_diff <= ASR_HUMAN_MAX_WORD_DIFF else "difficult"
    return status, round(cer, 4), n_diff


def score_asr_rows(df):
    results = [compare_transcripts(a, b) for a, b in zip(df["model_A_transcript"], df["model_B_transcript"])]
    statuses = [r[0] for r in results]
    df = df.copy()
    df["automatic_status"] = statuses
    df["status"] = statuses
    df["review_status"] = ["accepted" if s == "accepted" else "pending" for s in statuses]
    df["final_transcript"] = [a if s == "accepted" else "" for a, s in zip(df["model_A_transcript"], statuses)]
    df["cer"] = [r[1] for r in results]
    df["n_diff_words"] = [r[2] for r in results]
    df["second_model"] = "audar"
    df["changed_by"] = ""
    df["changed_at"] = ""
    df["change_log"] = "[]"
    return df


def blank_asr_rows(df, status, review_status):
    df = df.copy()
    for col, val in BLANK_META.items():
        df[col] = val
    df["status"] = status
    df["automatic_status"] = status
    df["review_status"] = review_status
    return df

Merges ASR rows with music/silence rows and writes `segments.csv` (16-column app format), `segments_full.csv`, `segments_asr.csv` and `transcripts/*.txt`.

In [ ]:
for vdir, job in asr_jobs.items():
    df_asr = score_asr_rows(job["df_asr"]) if len(job["df_asr"]) else job["df_asr"]
    df_non_asr = blank_asr_rows(job["df_non_asr"], "no_asr", "skipped") if len(job["df_non_asr"]) else job["df_non_asr"]
    df_skipped = blank_asr_rows(job["df_asr_skipped"], "not_run", "pending") if len(job["df_asr_skipped"]) else job["df_asr_skipped"]

    parts = [p for p in (df_asr, df_non_asr, df_skipped) if len(p) > 0]
    merged_df = pd.concat(parts, ignore_index=True).sort_values("_row_idx").reset_index(drop=True)

    df_out = merged_df[[c for c in APP_COLS if c in merged_df.columns]].copy()
    df_out.to_csv(vdir / "segments.csv", index=False, encoding="utf-8-sig")
    df_out.to_csv(vdir / "segments_asr.csv", index=False, encoding="utf-8-sig")

    full_df = merged_df.copy()
    s1_path = vdir / "segments_student1.csv"
    diag_path = vdir / "snr_diagnostics.csv"
    if s1_path.exists():
        s1_df = pd.read_csv(s1_path, dtype={"segment_id": str}).reset_index(drop=True)
        if len(s1_df) == len(full_df):
            if "status" in s1_df.columns:
                full_df["qc_status"] = s1_df["status"].values
            for c in [c for c in s1_df.columns if c not in full_df.columns]:
                full_df[c] = s1_df[c].values
    if diag_path.exists():
        d_df = pd.read_csv(diag_path, dtype={"segment_id": str}).reset_index(drop=True)
        if len(d_df) == len(full_df):
            for c in ["snr_energy_db", "snr_wada_db", "snr_disagree"]:
                if c in d_df.columns:
                    full_df[c] = d_df[c].values
    full_df.drop(columns=["_row_idx"], errors="ignore").to_csv(
        vdir / "segments_full.csv", index=False, encoding="utf-8-sig"
    )

    txt_dir = vdir / "transcripts"
    txt_dir.mkdir(parents=True, exist_ok=True)
    for r in df_asr.itertuples():
        txt_path = txt_dir / f"segment_{r.segment_id}.txt"
        if r.status == "accepted":
            txt_path.write_text(str(r.final_transcript), encoding="utf-8")
        else:
            txt_path.write_text(
                f"[{r.status}]\nModel A (Whisper): {r.model_A_transcript}\nModel B (Audar): {r.model_B_transcript}\n",
                encoding="utf-8",
            )

    print(f"\n[{vdir.name}] Status breakdown: {df_out['status'].value_counts().to_dict()}")
    print(f"  Saved: {vdir / 'segments.csv'} ({len(df_out.columns)} cols)")
    print(f"  Saved: {vdir / 'segments_full.csv'} ({len(full_df.columns) - 1} cols)")
    display(df_out[["segment_id", "label", "snr", "quality", "model_A_transcript", "model_B_transcript", "status"]].head(10))

print("\n✅ Phase 3 complete.")

## 6 · Phase 4 — Packaging
Builds the Step 7 summary table, the global manifests and zero-leakage `train`/`val`/`test` splits.

| Option | Possible values |
|---|---|
| `SPLIT_RATIOS` | `(0.70, 0.15, 0.15)` default — splits are only created with ≥ 3 videos |

In [ ]:
video_folders = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and (p / "segments_asr.csv").exists() and (p / "regions.csv").exists()
)
assert video_folders, "No video folders with ASR results found. Run Phase 3 first."

summary_rows, app_parts, full_parts = [], [], []

for vdir in video_folders:
    info_path = vdir / "info.json"
    info = json.loads(info_path.read_text(encoding="utf-8")) if info_path.exists() else {}
    reg_df = pd.read_csv(vdir / "regions.csv")
    seg_app = pd.read_csv(vdir / "segments.csv", dtype={"segment_id": str})
    app_parts.append(seg_app.assign(video_id=vdir.name))

    full_path = vdir / "segments_full.csv"
    if full_path.exists():
        full_df = pd.read_csv(full_path, dtype={"segment_id": str})
        if "video_id" not in full_df.columns:
            full_df.insert(0, "video_id", vdir.name)
        full_parts.append(full_df)

    reg_dur = reg_df.assign(d=reg_df["end"] - reg_df["start"]).groupby("label")["d"].sum() / 60.0
    asr_sub = seg_app[seg_app["label"].astype(str).str.lower().isin(["speech", "mixed"])]
    n_asr = max(1, len(asr_sub))
    st_counts = asr_sub["status"].value_counts().to_dict() if len(asr_sub) else {}
    mean_snr = pd.to_numeric(seg_app["snr"], errors="coerce").mean()

    summary_rows.append({
        "video_id": vdir.name,
        "title": info.get("title", vdir.name),
        "duration_min": round(float(reg_df["end"].max()) / 60.0, 2) if len(reg_df) else 0.0,
        "speech_min": round(float(reg_dur.get("speech", 0.0)), 2),
        "music_min": round(float(reg_dur.get("music", 0.0)), 2),
        "mixed_min": round(float(reg_dur.get("mixed", 0.0)), 2),
        "silence_min": round(float(reg_dur.get("silence", 0.0)), 2),
        "total_segments": len(seg_app),
        "asr_segments": len(asr_sub),
        "mean_snr_db": round(float(mean_snr), 2) if pd.notnull(mean_snr) else None,
        "clean_pct": round(100.0 * float((seg_app["quality"] == "clean").mean()), 1) if len(seg_app) else 0.0,
        "accepted_pct": round(100.0 * st_counts.get("accepted", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "human_choice_pct": round(100.0 * st_counts.get("human_choice", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "difficult_pct": round(100.0 * st_counts.get("difficult", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "license": info.get("license"),
        "url": info.get("url"),
    })

step7_df = pd.DataFrame(summary_rows)
step7_df.to_csv(DATA_ROOT / "corpus_summary_step7.csv", index=False, encoding="utf-8-sig")
print("📊 Step 7 corpus summary:")
display(step7_df)

pd.concat(app_parts, ignore_index=True).to_csv(
    DATA_ROOT / "corpus_manifest_app.csv", index=False, encoding="utf-8-sig"
)

if full_parts:
    manifest_full = pd.concat(full_parts, ignore_index=True)
    if "duration" not in manifest_full.columns:
        manifest_full["duration"] = (
            pd.to_numeric(manifest_full["end"]) - pd.to_numeric(manifest_full["start"])
        ).round(3)
    manifest_full.to_csv(DATA_ROOT / "corpus_manifest_full.csv", index=False, encoding="utf-8-sig")
    splits = split_corpus(manifest_full, out_dir=DATA_ROOT / "splits", ratios=SPLIT_RATIOS)
    print("Split status:", splits.get("status"))

Zips the corpus (without the large intermediate audio files) into `/kaggle/working/corpus_final_output.zip`.

In [ ]:
zip_path = WORK_ROOT / "corpus_final_output.zip"
skip_names = {"audio.wav", "audio_conditioned.flac", "video.mp4"}

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(DATA_ROOT.rglob("*")):
        if f.is_file() and f.name not in skip_names and not f.name.endswith(".zip"):
            zf.write(f, arcname=f.relative_to(WORK_ROOT))

print(f"📦 Final archive ready: {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MB)")
print(f"   All files remain under: {DATA_ROOT.resolve()}")